## EDA And Feature Engineering Of Google Play Store Dataset

#### 1) Problem statement:
- Today, 1.85 million different apps are available for users to download.
- Android users have even more from which to choose, with 2.56 million available through the Google Play Store.
- These apps have come to play a huge role in the way we live our lives today.
- Our Objective is to find:
  * The Most Popular Category
  * The App with largest number of installs
  * The App with largest size etc.

## Feature Information
1. App :- Name of the App
2. Category :- Category under which the App falls.
3. Rating :- Application's rating on playstore
4. Reviews :- Number of reviews of the App.
5. Size :- Size of the App.
6. Install :- Number of Installs of the App
7. Type :- If the App is free/paid
8. Price :- Price of the app (0 if it is Free)
9. Content Rating :- Appropiate Target Audience of the App.
10. Genres:- Genre under which the App falls.
11. Last Updated :- Date when the App was last updated
12. Current Ver :- Current Version of the Application
13. Android Ver :- Minimum Android Version required to run the App

### Importing required libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

import warnings
warnings.filterwarnings("ignore")

### Loading dataset

In [ ]:
df = pd.read_csv("Google-Playstore-Data/googleplaystore.csv")

### Overview of the dataset

In [ ]:
df.shape

In [ ]:
pd.set_option('display.max_columns', None)

In [ ]:
df.head()

In [ ]:
df.columns

In [ ]:
df.drop(labels='Unnamed: 0',
       axis=1,
       inplace=True)

### Data Cleaning

In [ ]:
# checking for missing values
df.isnull().any()

In [ ]:
# finding total no. of missing values
df.isnull().sum()

In [ ]:
# visualization of missing values with help of bar charts
plt.figure(figsize=(20,4))
bars = plt.bar(x=df.isnull().sum().index,
       height=df.isnull().sum().values,
       edgecolor='black',
       width=0.4,
       color='red')
plt.bar_label(bars,
              labels=df.isnull().sum().values,
              label_type='edge')
plt.xlabel('Name of the Features')
plt.ylabel('Missing Value Count')
plt.title('VISUALIZATION OF MISSING VALUES USING BAR GRAPH')
plt.savefig("reports/missing_value_analysis.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# brief overview of dataset
df.info()

### So, we have to clean the numerical columns and convert them into integer or float type, which ever is required

### As, we can see, 'review' should be an integer type column. Lets clean that column.

In [ ]:
df.head()

In [ ]:
df['Reviews'].unique()

In [ ]:
# checking for numerical strings
print(df['Reviews'].str.isnumeric().sum())

In [ ]:
# actual length of the series data
len(df['Reviews'])

In [ ]:
len(df['Reviews']) - (df['Reviews'].str.isnumeric().sum())

#### So, there is one value in the 'Review' column which is not numeric

In [ ]:
# checking that non-numeric element
df[~df['Reviews'].str.isnumeric()]

In [ ]:
# deleting that row
df_filtered = df.drop(df.index[10472])
df_filtered = df_filtered.reset_index(drop=True)

In [ ]:
# checking
df_filtered[~df_filtered['Reviews'].str.isnumeric()]

In [ ]:
# converting datatype to int
df_filtered['Reviews'] = df_filtered['Reviews'].astype(int)

In [ ]:
df_filtered.head(2)

### As we can see, 'size' should be a float type column. Lets clean that column.

In [ ]:
# checking for unique values
df_filtered['Size'].unique()

#### So, we can conclude that, there are mainly 5 types of format available. e.g.
- '19M'
- '3.1M'
- '899k'
- '8.5k'
- 'Varies with device'
#### We are going to convert all numerical formats into 'k'-scale:
- '19M' --> 19000
- '3.1M' --> 3100
- '899k' --> 899
- '8.5k' --> 8.5
#### We can replce 'Varies with device' with 'np.nan'

In [ ]:
def convert(x):
    if 'M' in x:
        x = float(x.replace('M',''))*1000
        return x
    elif 'k' in x:
        x = float(x.replace('k',''))
        return x
    else:
        return np.nan

In [ ]:
df_filtered['Size'] = df_filtered['Size'].apply(lambda x: convert(x))

In [ ]:
df_filtered.head()

### As we can see, 'Installs' should be an integer type column. Let us clean that column.

In [ ]:
chars_to_remove = [',','+']
for item in chars_to_remove:
    df_filtered['Installs'] = df_filtered['Installs'].str.replace(item, '')

In [ ]:
df_filtered['Installs'] = df_filtered['Installs'].astype(int)

In [ ]:
df_filtered

In [ ]:
# brief description of the dataset
df_filtered.info()

### As we can see, the 'Price' should be a float column. Lets clean that column.

In [ ]:
# checking unique values
df_filtered['Price'].unique()

In [ ]:
df_filtered['Price'].apply(lambda x: float(x.strip().replace('$',''))).unique()

In [ ]:
df_filtered['Price'] = df_filtered['Price'].apply(lambda x: float(x.strip().replace('$','')))

In [ ]:
df_filtered.info()

### checking for duplicate apps and deleting them, because duplicate apps will not provide any useful information during the anlysis.

In [ ]:
df_filtered['App'].nunique()

In [ ]:
# checking duplicated apps
df_filtered.duplicated(subset=['App']).value_counts()

In [ ]:
df_filtered = df_filtered.drop_duplicates(subset=['App'], keep='first')

### converting the 'Last Updated' column into datetime format and also storing the date, month and year into separate columns.

In [ ]:
df_filtered['Last Updated'].unique()

In [ ]:
pd.to_datetime(df_filtered['Last Updated'],
              format='%d-%b-%y')

In [ ]:
df_filtered['Last Updated'] = pd.to_datetime(df_filtered['Last Updated'],
                                             format='%d-%b-%y')
df_filtered['Day_updated'] = df_filtered['Last Updated'].dt.day
df_filtered['Month_updated'] = df_filtered['Last Updated'].dt.month
df_filtered['Year_updated'] = df_filtered['Last Updated'].dt.year

In [ ]:
df_filtered.info()

In [ ]:
df_filtered.drop(labels='Last Updated',
                axis=1,
                inplace=True)

### Exploratory Data Analysis

In [ ]:
categorical_feature = [feature for feature in df_filtered.columns if (df_filtered[feature].dtype == 'str')]
numeric_features = [feature for feature in df_filtered.columns if feature not in categorical_feature]

# print columns
print("We have {} numerical features: {}".format(len(numeric_features), numeric_features))
print("We have {} categorical features: {}".format(len(categorical_feature), categorical_feature))

#### Categorical Columns

In [ ]:
# Proportion of count data in each categorical column
for col in categorical_feature:
    print('_________________________________')
    print(df[col].value_counts(normalize=True)*100)

In [ ]:
# # Proportion of count data on categorical columns
plt.figure(figsize=(15, 6))
plt.suptitle('Univariate Analysis of Categorical Features', fontsize=20, fontweight='bold', alpha=0.8, y=1)
category = [ 'Type', 'Content Rating']

for i,col in enumerate(category):
    plt.subplot(1, 2, i+1)
    sns.countplot(data=df_filtered[col], palette="Set2")
    plt.xlabel(col)
    plt.xticks(rotation=45)
    plt.tight_layout()
plt.savefig("reports/Univariate_Analysis_of_Categorical_Features.png", dpi=300, bbox_inches="tight");

#### Numerical Columns

In [ ]:
# Proportion of count data on numerical columns
plt.figure(figsize=(15, 6))
plt.suptitle('Univariate Analysis of Numerical Features', fontsize=20, fontweight='bold', alpha=0.8, y=1)

for i,col in enumerate(numeric_features):
    plt.subplot(2, 4, i+1)
    sns.kdeplot(data=df_filtered[col],shade=True, color='r')
    plt.xlabel(col)
    plt.xticks(rotation=90)
    plt.tight_layout()
plt.savefig("reports/Univariate_Analysis_of_Numerical_Features.png", dpi=300, bbox_inches="tight");

### Observation
- Rating and year both are left skewed
- Reviews, Size, Installs, Price are right skewed

# Which is the most popular app category?

In [ ]:
labels = df_filtered['Category'].value_counts().index
values = df_filtered['Category'].value_counts().values
explode = [0.2]
for _ in range(len(labels)-1):
    explode.append(0)

plt.figure(figsize=(10, 10))
plt.pie(values,explode=explode,labels=labels,autopct="%1.1f%%",shadow=True)
plt.title('Visualization of most popular category with the help of Pie Chart', fontsize=20, fontweight='bold', loc='center', alpha=0.8)
plt.savefig("reports/most_popular_categories.png", dpi=300, bbox_inches="tight")
plt.show()

## Observation:
1. The most popular category is "Family"
2. Three least popular categories are: "Beauty", "Comics" and "Parenting"

# What are the top 10 most popular app categories?

In [ ]:
category_df = pd.DataFrame({'App_categories': df_filtered['Category'].value_counts().index,
                           'Count': df_filtered['Category'].value_counts().values,
                           'Percentage': df_filtered['Category'].value_counts(normalize=True).values.round(4)*100})
category_df.head(10)

In [ ]:
x_labels = category_df.loc[:9, 'App_categories']
values = category_df.loc[:9, 'Count']
colors = plt.get_cmap("tab10").colors[:len(x_labels)]
labels = [f"{round(p,2)}%" for p in category_df.loc[:9, 'Percentage']]

plt.figure(figsize=(15, 5))
bars = plt.bar(x=x_labels,
       height=values,
       edgecolor='black',
       width=0.4,
       color=colors)
plt.bar_label(bars,
              labels=labels,
              label_type='edge')
plt.xticks(rotation=45)
plt.title('Visualization of TOP 10 most popular app category with the help of Bar Chart',
          fontsize=20, fontweight='bold', loc='center', alpha=0.8)
plt.savefig("reports/Tpo_10_popular_apps.png", dpi=300, bbox_inches="tight");
plt.show()

# Which category has largest number of installation?

In [ ]:
install_df = pd.DataFrame({'category': df_filtered.groupby(by='Category')['Installs'].aggregate('sum').index,
                          'Total_installs': df_filtered.groupby(by='Category')['Installs'].aggregate('sum').values})
install_df.sort_values(by='Total_installs', ascending=False, inplace=True)
install_df.reset_index(drop=True, inplace=True)
install_df.head(10)

In [ ]:
labels = list(install_df.loc[0:9, 'category'])
values = list(install_df.loc[0:9, 'Total_installs'])
explode = [0.2]
for _ in range(len(labels)-1):
    explode.append(0)

plt.figure(figsize=(10, 10))
plt.pie(values,explode=explode,labels=labels,autopct="%1.1f%%",shadow=True)
plt.title('Visualization of most installed category with the help of Pie Chart', fontsize=20, fontweight='bold', loc='center', alpha=0.8)
plt.savefig("reports/Most_installed_categories.png", dpi=300, bbox_inches="tight");
plt.show()

# Which are the top 5 most installed Apps in each of the 10 most popular categories?

In [ ]:
# 10 most popular categories
popular_cat = list(category_df.loc[:9,'App_categories'])

In [ ]:
data = {"Category":[], "App": [], "Installs": []}

for cat in popular_cat:
    print('____________CATEGORY: {}________________'.format(cat))
    print(df_filtered[df_filtered['Category']==cat].groupby(by='App')['Installs'].sum().sort_values(ascending=False)[:5])
    app_list = list(df_filtered[df_filtered['Category']==cat].groupby(by='App')['Installs'].sum().sort_values(ascending=False)[:5].index)
    values = list(df_filtered[df_filtered['Category']==cat].groupby(by='App')['Installs'].sum().sort_values(ascending=False)[:5].values)
    for i,app in enumerate(app_list):
        data["Category"].append(cat)
        data["App"].append(app)
        data["Installs"].append(values[i])

In [ ]:
new_df = pd.DataFrame(data)
new_df

In [ ]:
import plotly.io as pio
pio.renderers.default = "vscode"

In [ ]:
fig = px.sunburst(new_df, 
                  path=["Category", "App"],
                  values="Installs",
                  color="Category")
fig.update_layout( width=900, height=900,
                 title={ "text": "App Installs by Category", 
                         "x": 0.5, # center 
                         "y": 0.95, # vertical position 
                         "xanchor": "center", 
                         "yanchor": "top", 
                         "font": dict(size=24, color="darkblue", family="Arial")
                       }
                 )
fig.write_image("reports/app_installed_by_category.png",scale=2)
fig.show()

# How many apps are there on Google Play Store which get 5 ratings?

In [ ]:
df_filtered[df_filtered['Rating']==5].reset_index(drop=True)

In [ ]:
print("No. of apps: {}".format(len(df_filtered[df_filtered['Rating']==5])))